# Phase 0 — Raw Audit and First-Stage Standardization

This notebook performs the raw audit and first-stage standardization of the finance ABSA dataset.

The goals of this phase are:

1. load the raw CSV safely;
2. verify the schema and basic integrity of the dataset;
3. parse the `Decisions` field into structured aspect-sentiment annotations;
4. identify data quality issues such as malformed labels, duplicates, and suspicious ordering;
5. generate a first-stage audited CSV for downstream processing;
6. produce an audit summary to confirm whether the dataset is ready for the next stage.

In [1]:
import pandas as pd
import numpy as np
import json
import ast
import re
from pathlib import Path
from collections import Counter, defaultdict
from pprint import pprint

In [2]:
RAW_PATH = Path("SEntFiN-v1.1.csv")
OUT_DIR = Path("outputs_phase0")
OUT_DIR.mkdir(parents=True, exist_ok=True)

AUDITED_CSV_PATH = OUT_DIR / "finance_raw_audited.csv"
AUDIT_JSON_PATH = OUT_DIR / "audit_report.json"
DUPLICATE_CSV_PATH = OUT_DIR / "duplicate_report.csv"
CHECKLIST_CSV_PATH = OUT_DIR / "audit_checklist.csv"

## Step 1. Load the raw CSV

In this step, we load the raw dataset as immutable input.  
The raw CSV is treated as the original source and will not be modified directly.

In [3]:
df_raw = pd.read_csv(RAW_PATH)
print("Raw shape:", df_raw.shape)
df_raw.head()

Raw shape: (10753, 4)


,S No.,Title,Decisions,Words
0,1,SpiceJet to issue 6.4 crore warrants to promoters,"{""SpiceJet"": ""neutral""}",8
1,2,MMTC Q2 net loss at Rs 10.4 crore,"{""MMTC"": ""neutral""}",8
2,3,"Mid-cap funds can deliver more, stay put: Experts","{""Mid-cap funds"": ""positive""}",8
3,4,Mid caps now turn into market darlings,"{""Mid caps"": ""positive""}",7
4,5,"Market seeing patience, if not conviction: Pra...","{""Market"": ""neutral""}",8


## Step 2. Audit the raw schema

We first check whether the dataset contains the expected columns and whether any critical fields are missing.

In [4]:
expected_columns = ["S No.", "Title", "Decisions", "Words"]
actual_columns = list(df_raw.columns)

schema_ok = actual_columns == expected_columns

print("Expected columns:", expected_columns)
print("Actual columns:  ", actual_columns)
print("Schema OK:", schema_ok)

missing_title = df_raw["Title"].isna().sum()
missing_decisions = df_raw["Decisions"].isna().sum()

print("Missing Title:", missing_title)
print("Missing Decisions:", missing_decisions)

Expected columns: ['S No.', 'Title', 'Decisions', 'Words']
Actual columns:   ['S No.', 'Title', 'Decisions', 'Words']
Schema OK: True
Missing Title: 0
Missing Decisions: 0


## Step 3. Parse the `Decisions` field

The `Decisions` column is expected to contain a dictionary-like string mapping aspect/entity terms to sentiment labels.

Example:
{"Silver": "positive", "gold": "positive"}

This step converts the raw string into a structured Python dictionary.

In [5]:
ALLOWED_LABELS = {"positive", "negative", "neutral"}

def safe_parse_decisions(x):
    if pd.isna(x):
        return None, "missing"
    if isinstance(x, dict):
        return x, None
    try:
        parsed = ast.literal_eval(x)
        if not isinstance(parsed, dict):
            return None, "not_dict"
        return parsed, None
    except Exception:
        return None, "parse_error"

In [6]:
parsed_records = []
parse_issues = []

for idx, row in df_raw.iterrows():
    parsed, err = safe_parse_decisions(row["Decisions"])
    parsed_records.append(parsed)
    parse_issues.append(err)

df = df_raw.copy()
df["parsed_decisions"] = parsed_records
df["parse_issue"] = parse_issues

print(df["parse_issue"].value_counts(dropna=False))

parse_issue
None    10753
Name: count, dtype: int64


## Step 4. Validate sentiment labels

We now verify whether all parsed aspect labels belong to the allowed sentiment set:
positive, negative, neutral.

In [7]:
def extract_invalid_labels(d):
    if d is None:
        return []
    invalid = []
    for k, v in d.items():
        if str(v).strip().lower() not in ALLOWED_LABELS:
            invalid.append((k, v))
    return invalid

df["invalid_labels"] = df["parsed_decisions"].apply(extract_invalid_labels)
df["num_invalid_labels"] = df["invalid_labels"].apply(len)

print("Rows with invalid labels:", (df["num_invalid_labels"] > 0).sum())
df.loc[df["num_invalid_labels"] > 0, ["S No.", "Title", "Decisions", "invalid_labels"]].head(10)

Rows with invalid labels: 0


,S No.,Title,Decisions,invalid_labels


## Step 5. Light normalization

This phase applies only safe first-stage normalization:
- trim leading/trailing spaces,
- normalize repeated whitespace,
- normalize sentiment label casing.

No aggressive text rewriting is performed at this stage.

In [8]:
def normalize_text(text):
    if pd.isna(text):
        return text
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

def normalize_decision_dict(d):
    if d is None:
        return None
    out = {}
    for k, v in d.items():
        kk = str(k).strip()
        vv = str(v).strip().lower()
        out[kk] = vv
    return out

df["title_norm"] = df["Title"].apply(normalize_text)
df["decisions_norm"] = df["parsed_decisions"].apply(normalize_decision_dict)

## Step 6. Build a basic dataset profile

This step summarizes:
- number of headlines,
- number of aspect-sentiment pairs,
- distribution of sentiment labels,
- number of multi-aspect headlines.

In [9]:
def num_aspects(d):
    if d is None:
        return 0
    return len(d)

df["num_aspects"] = df["decisions_norm"].apply(num_aspects)

all_labels = []
for d in df["decisions_norm"]:
    if d is not None:
        all_labels.extend(list(d.values()))

label_counter = Counter(all_labels)

num_headlines = len(df)
num_pairs = sum(df["num_aspects"])
num_multi_aspect = (df["num_aspects"] > 1).sum()

print("Number of headlines:", num_headlines)
print("Number of aspect-sentiment pairs:", num_pairs)
print("Multi-aspect headlines:", num_multi_aspect)
print("Sentiment distribution:")
pprint(label_counter)

Number of headlines: 10753
Number of aspect-sentiment pairs: 14409
Multi-aspect headlines: 2850
Sentiment distribution:
Counter({'neutral': 5515, 'positive': 5075, 'negative': 3819})


## Step 7. Check whether the raw order appears safe for direct splitting

The dataset must not be split directly by raw file order unless the order is shown to be approximately random.
This step inspects local windows of the dataset to identify suspicious label clustering.

In [10]:
def headline_signature(d):
    if d is None or len(d) == 0:
        return "empty"
    labels = sorted(set(d.values()))
    return "+".join(labels)

df["label_signature"] = df["decisions_norm"].apply(headline_signature)

window_size = 500
window_stats = []

for start in range(0, len(df), window_size):
    end = min(start + window_size, len(df))
    chunk = df.iloc[start:end]
    sig_counts = chunk["label_signature"].value_counts().to_dict()
    window_stats.append({
        "start": start,
        "end": end,
        "size": len(chunk),
        "top_signatures": sig_counts
    })

window_stats[:3]

[{'start': 0,
  'end': 500,
  'size': 500,
  'top_signatures': {'neutral': 487,
   'positive': 6,
   'negative+neutral': 3,
   'neutral+positive': 2,
   'negative+positive': 1,
   'negative': 1}},
 {'start': 500,
  'end': 1000,
  'size': 500,
  'top_signatures': {'neutral': 477,
   'positive': 11,
   'negative': 5,
   'neutral+positive': 3,
   'negative+positive': 2,
   'negative+neutral': 2}},
 {'start': 1000,
  'end': 1500,
  'size': 500,
  'top_signatures': {'neutral': 448,
   'positive': 39,
   'neutral+positive': 5,
   'negative': 5,
   'negative+positive': 1,
   'negative+neutral+positive': 1,
   'negative+neutral': 1}}]

## Step 8. Audit duplicated or near-duplicated headlines

We check:
1. exact duplicated titles;
2. exact duplicated (title, decisions) pairs;
3. duplicated titles with conflicting labels.

In [11]:
df["title_key"] = df["title_norm"].str.lower()

dup_title_mask = df.duplicated(subset=["title_key"], keep=False)
dup_exact_mask = df.duplicated(subset=["title_key", "Decisions"], keep=False)

dup_titles = df.loc[dup_title_mask].copy()
dup_exact = df.loc[dup_exact_mask].copy()

print("Duplicated titles:", dup_title_mask.sum())
print("Duplicated exact rows:", dup_exact_mask.sum())

Duplicated titles: 134
Duplicated exact rows: 110


In [12]:
conflict_rows = []

for title_key, group in df.groupby("title_key"):
    if len(group) <= 1:
        continue
    unique_decisions = group["Decisions"].dropna().unique()
    if len(unique_decisions) > 1:
        for _, row in group.iterrows():
            conflict_rows.append({
                "title_key": title_key,
                "S No.": row["S No."],
                "Title": row["Title"],
                "Decisions": row["Decisions"]
            })

df_conflict = pd.DataFrame(conflict_rows)
print("Duplicated titles with conflicting decisions:", len(df_conflict))
df_conflict.head()

Duplicated titles with conflicting decisions: 24


,title_key,S No.,Title,Decisions
0,asian shares flat as investors digest greek deal,5695,Asian shares flat as investors digest Greek deal,"{""Asian shares"": ""neutral""}"
1,asian shares flat as investors digest greek deal,8950,Asian shares flat as investors digest Greek deal,"{""Asian shares"": ""negative""}"
2,carl icahn says his firm sold remainder of net...,6246,Carl Icahn says his firm sold remainder of Net...,"{""Netflix"": ""neutral""}"
3,carl icahn says his firm sold remainder of net...,9400,Carl Icahn says his firm sold remainder of Net...,"{""Netflix"": ""negative""}"
4,crude oil futures down on weak asian cues,3499,Crude oil futures down on weak Asian cues,"{""Crude oil futures"": ""negative""}"


In [13]:
duplicate_export = df.loc[dup_title_mask | dup_exact_mask].copy()
duplicate_export.to_csv(DUPLICATE_CSV_PATH, index=False)
print("Saved duplicate report to:", DUPLICATE_CSV_PATH)

Saved duplicate report to: outputs_phase0\duplicate_report.csv


## Step 9. Create the first-stage audited CSV

This file keeps the original headline-level granularity, while adding parsed and normalized information required for downstream standardization.

In [14]:
def decisions_to_json_string(d):
    if d is None:
        return ""
    return json.dumps(d, ensure_ascii=False, sort_keys=True)

df_audited = pd.DataFrame({
    "raw_id": df["S No."],
    "title_raw": df["Title"],
    "title_norm": df["title_norm"],
    "words_raw": df["Words"],
    "decisions_raw": df["Decisions"],
    "decisions_norm_json": df["decisions_norm"].apply(decisions_to_json_string),
    "num_aspects": df["num_aspects"],
    "label_signature": df["label_signature"],
    "parse_issue": df["parse_issue"].fillna("ok"),
    "num_invalid_labels": df["num_invalid_labels"]
})

df_audited.to_csv(AUDITED_CSV_PATH, index=False)
print("Saved audited CSV to:", AUDITED_CSV_PATH)
df_audited.head()

Saved audited CSV to: outputs_phase0\finance_raw_audited.csv


,raw_id,title_raw,title_norm,words_raw,decisions_raw,decisions_norm_json,num_aspects,label_signature,parse_issue,num_invalid_labels
0,1,SpiceJet to issue 6.4 crore warrants to promoters,SpiceJet to issue 6.4 crore warrants to promoters,8,"{""SpiceJet"": ""neutral""}","{""SpiceJet"": ""neutral""}",1,neutral,ok,0
1,2,MMTC Q2 net loss at Rs 10.4 crore,MMTC Q2 net loss at Rs 10.4 crore,8,"{""MMTC"": ""neutral""}","{""MMTC"": ""neutral""}",1,neutral,ok,0
2,3,"Mid-cap funds can deliver more, stay put: Experts","Mid-cap funds can deliver more, stay put: Experts",8,"{""Mid-cap funds"": ""positive""}","{""Mid-cap funds"": ""positive""}",1,positive,ok,0
3,4,Mid caps now turn into market darlings,Mid caps now turn into market darlings,7,"{""Mid caps"": ""positive""}","{""Mid caps"": ""positive""}",1,positive,ok,0
4,5,"Market seeing patience, if not conviction: Pra...","Market seeing patience, if not conviction: Pra...",8,"{""Market"": ""neutral""}","{""Market"": ""neutral""}",1,neutral,ok,0


## Step 10. Save the audit report

The audit report summarizes the integrity and risks of the raw dataset before moving to the next phase.

In [15]:
audit_report = {
    "raw_shape": list(df_raw.shape),
    "schema_ok": bool(schema_ok),
    "missing_title": int(missing_title),
    "missing_decisions": int(missing_decisions),
    "rows_with_parse_issue": int(df["parse_issue"].notna().sum()),
    "rows_with_invalid_labels": int((df["num_invalid_labels"] > 0).sum()),
    "num_headlines": int(num_headlines),
    "num_aspect_pairs": int(num_pairs),
    "num_multi_aspect_headlines": int(num_multi_aspect),
    "label_distribution": dict(label_counter),
    "duplicated_titles": int(dup_title_mask.sum()),
    "duplicated_exact_rows": int(dup_exact_mask.sum()),
    "duplicated_conflicting_rows": int(len(df_conflict)),
    "raw_order_safe_for_direct_split": False
}

with open(AUDIT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(audit_report, f, ensure_ascii=False, indent=2)

print("Saved audit report to:", AUDIT_JSON_PATH)
pprint(audit_report)

Saved audit report to: outputs_phase0\audit_report.json
{'duplicated_conflicting_rows': 24,
 'duplicated_exact_rows': 110,
 'duplicated_titles': 134,
 'label_distribution': {'negative': 3819, 'neutral': 5515, 'positive': 5075},
 'missing_decisions': 0,
 'missing_title': 0,
 'num_aspect_pairs': 14409,
 'num_headlines': 10753,
 'num_multi_aspect_headlines': 2850,
 'raw_order_safe_for_direct_split': False,
 'raw_shape': [10753, 4],
 'rows_with_invalid_labels': 0,
 'rows_with_parse_issue': 0,
 'schema_ok': True}


## Step 11. Evaluate whether Phase 0 is complete

This checklist determines whether the dataset is ready to move to the next stage.

In [16]:
checklist = [
    ("Schema matches expected columns", schema_ok),
    ("No missing titles", missing_title == 0),
    ("No missing decisions", missing_decisions == 0),
    ("All decisions parsed successfully", df["parse_issue"].isna().all()),
    ("No invalid sentiment labels", (df["num_invalid_labels"] == 0).all()),
    ("Audited CSV exported", AUDITED_CSV_PATH.exists()),
    ("Audit report exported", AUDIT_JSON_PATH.exists()),
    ("Duplicate report exported", DUPLICATE_CSV_PATH.exists()),
]

df_checklist = pd.DataFrame(checklist, columns=["check_item", "passed"])
df_checklist.to_csv(CHECKLIST_CSV_PATH, index=False)

display(df_checklist)

all_passed = df_checklist["passed"].all()
print("PHASE 0 COMPLETE:", all_passed)

,check_item,passed
0,Schema matches expected columns,True
1,No missing titles,True
2,No missing decisions,True
3,All decisions parsed successfully,True
4,No invalid sentiment labels,True
5,Audited CSV exported,True
6,Audit report exported,True
7,Duplicate report exported,True


PHASE 0 COMPLETE: True


## Phase 0 conclusion

This phase is considered complete if:

- the raw CSV can be safely loaded;
- the schema is verified;
- the `Decisions` column is parsed successfully;
- sentiment labels are validated;
- major duplicate patterns are logged;
- a first-stage audited CSV is exported;
- an audit report is generated.

If all checklist items pass, the dataset is ready for Phase 1:
canonical normalization and split preparation.